In [ ]:
# Import python packages
import streamlit as st

import pandas as pd
import re
from typing import List
from typing import Tuple


from pathlib import Path
from io import BytesIO

# scaffold_dbt.py
import argparse, os, textwrap, sys
import itertools

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
def needs_quotes(name: str) -> bool:
    """
    Decide si un identificador SQL necesita comillas dobles.
    - Si ya empieza y acaba con comillas dobles -> NO necesita (ya las tiene).
    - Si cumple el patrón de identificador no citado válido -> NO necesita.
    - En cualquier otro caso -> SÍ necesita.
    """
    if name.startswith('"') and name.endswith('"'):
        return False # Si ya viene entre comillas dobles => False

    # identificador estándar válido sin comillas
    return not re.match(r'^[A-Z_a-z][A-Za-z0-9_\$]*$', name or '')


def quoted(name: str) -> str:
    """
    Devuelve el identificador SQL, entre comillas dobles si es necesario.
    Convierte el identificador a mayúsculas si no es necesario entrecomillarlo.
    """
    return f'"{name}"' if needs_quotes(name) else name.upper()

def quoted_src_cls(name: str) -> str:
    """
    Devuelve el identificador SQL, entre comillas siempre
    """
    return f'"{name}"'
    
def parse_bool(val) -> bool:
    """
    Parsea un valor booleano desde varios formatos comunes.
    Valores verdaderos: True, 1, 1.0, 'TRUE', 'T', '1', 'S', 'SI', 'YES', 'Y'
    Valores falsos: cualquier otro valor, incluyendo None, NaN o cadena vacía.
    Maneja columnas float64 de pandas donde True se convierte a 1.0 por presencia de NaN.
    """
    if val is None:
        return False
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)):
        try:
            return val == 1 and val == val  # val==val descarta NaN
        except Exception:
            return False
    return str(val).strip().upper() in ('TRUE','T','1','S','SI','YES','Y')

def write_text(rel_path: str, content: str, overwrite: bool = True):
    target = f"{root}/{rel_path}"
    session.file.put_stream(BytesIO(content.encode("utf-8")), target, overwrite=overwrite, auto_compress=False)

def extract_asociados_from_bronze(df: pd.DataFrame,
                                  db_col: str = "BBDD_SRC_NAME",
                                  schm_col: str = "SCHM_SRC_NAME",
                                  v_bronze_db='DEV_BRONZE',
                                  v_starts='BRONZE_',
                                  v_ends = '_P') -> list:
    """
    Extrae la lista de asociados únicos desde los esquemas que pertenecen a la base de datos BRONZE.
    """
    if df.empty or db_col not in df or schm_col not in df:
        return []

    mask = df[db_col].str.upper().eq(str(v_bronze_db).upper()) # Nos quedamos con as filas donde la columna db_col coincide con el nombre de la BD BRONZE
    schms = df.loc[mask, schm_col].dropna().astype(str).str.strip().unique().tolist() # Lista de esquemas únicos

    asociados = {extract_string_part(s, ends=v_ends, starts=v_starts) for s in schms}
    asociados = sorted(x for x in asociados if x)
    return asociados
    
def yml_list_str(items):
    """
    Devuelve una representación en cadena de una lista para YAML,
    con cada elemento entre comillas simples.
    """
    return "[" + ", ".join(f"'{x}'" for x in items) + "]"


def extract_string_part(s: str, ends: str = '', starts: str = '') -> str:
    """
    Extrae la parte central de una cadena eliminando un prefijo y un sufijo opcionales.
    """
    su = (s or "").strip().upper()
    if ends and su.endswith(ends.upper()):
        su = su[:-len(ends)]
    if starts and su.startswith(starts.upper()):
        su = su[len(starts):]
    return su.lower()
    

def jinja_sql_str(expr: str) -> str:
    """
    Envuelve una expresión SQL en comillas simples para Jinja y
    escapa comillas simples internas (' -> '').
    """
    s = str(expr or "")
    return "'" + s.replace("'", "''") + "'"


# COALESCE
def _dtype_base(dtype: str) -> str:
    """
    Recogemos el Dtype asignado (previo a '(')
    """
    if not dtype:
        return "VARCHAR"
    return dtype.strip().upper().split("(")[0].strip()

def _default_literal_for_dtype(dtype: str) -> str | None:
    """
    Definimos el valor coalesce según el dtype
    """
    b = _dtype_base(dtype)
    if b in ("VARCHAR","STRING","TEXT","CHAR","CHARACTER"):
        return "'N/I'"
    if b in ("INT","INTEGER","BIGINT","SMALLINT","TINYINT","NUMBER","NUMERIC","DECIMAL","FLOAT","DOUBLE","REAL"):
        return "0"
    if b == "DATE":
        return "DATE('1900-01-01')"
    if b == "TIMESTAMP_NTZ":
        return "TO_TIMESTAMP_NTZ('1900-01-01')"
    if b == "TIMESTAMP_LTZ":
        return "TO_TIMESTAMP_LTZ('1900-01-01')"
    if b == "TIMESTAMP_TZ":
        return "TO_TIMESTAMP_TZ('1900-01-01')"
    if b == "TIME":
        return "TO_TIME('00:00:00')"
    if b == "BOOLEAN":
        return "FALSE"
    return None  # sin default definido


def safe_str(v) -> str:
    if v is None or pd.isna(v):
        return ""
    return str(v).strip()

def normalize_default_literal(default_raw, dtype: str) -> str:
    """
    - Convierte NaN -> ''
    - Para enteros: 1900.0 -> 1900
    - Respeta expresiones SQL/Jinja (DATE(...), TO_TIMESTAMP..., {{...}})
    - Para texto: auto-quote si no viene ya quoted
    """
    s = safe_str(default_raw)
    if not s:
        return ""

    b = _dtype_base(dtype)

    if s.startswith(("'", '"')) or s.startswith("{{") or "(" in s: # Si parece expresión SQL/Jinja, la respetamos
        return s

    if b in ("INT","INTEGER","BIGINT","SMALLINT","TINYINT"): # en tipos enteros elimina .0 si procede
        try:
            f = float(s)
            if f.is_integer():
                return str(int(f))
            return str(f)
        except:
            return s

    if b in ("NUMBER","NUMERIC","DECIMAL","FLOAT","DOUBLE","REAL"): # NUMBER genérico: si viene con .0, lo eliminamos también
        if re.fullmatch(r"-?\d+\.0", s):
            return s[:-2]
        return s

    # Texto: auto-quote si viene sin comillas
    if b in ("VARCHAR","STRING","TEXT","CHAR","CHARACTER"):
        return f"'{s}'"

    return s


def apply_pk_coalesce(expr: str, dtype: str, default_lit_override: str | None = None) -> str:
    default_lit = default_lit_override or _default_literal_for_dtype(dtype)
    if not default_lit:
        return expr

    b = _dtype_base(dtype)

    if (expr or "").strip().upper() == "NULL":  # Si expr es literalmente NULL, devolvemos directamente el default
        return default_lit

    if b in ("VARCHAR","STRING","TEXT","CHAR","CHARACTER"):
        return f"COALESCE(NULLIF(TRIM(({expr})), ''), {default_lit})"

    return f"COALESCE(({expr}), {default_lit})"


def db_to_env_dbs(db: str) -> Tuple[str, str]:
    """
    A partir del DDBB 'DEV_BRONZE' / 'PROD_BRONZE' devuelve siempre: (dev_db, prod_db).

    Ej:
      'DEV_BRONZE'  -> ('DEV_BRONZE',  'PROD_BRONZE')
      'PROD_BRONZE' -> ('DEV_BRONZE',  'PROD_BRONZE')

    Si no matchea DEV_/PROD_, devuelve (UPPER(db), UPPER(db)).
    Si ya contiene Jinja ({{ o {% ), devuelve (db, db) tal cual.
    """
    if db is None:
        return ("", "")

    s = str(db).strip()
    if "{{" in s or "{%" in s:
        return (s, s)

    s_up = s.upper()

    m = re.match(r"^(DEV|PROD)_(.+)$", s_up)
    if not m:
        return (s_up, s_up)

    suffix = m.group(2)  # p.ej. BRONZE, SILVER, GOLD, SUPPORT, GOVERNANCE_DB...
    dev_db = f"DEV_{suffix}"
    prod_db = f"PROD_{suffix}"
    return (dev_db, prod_db)

In [ ]:
root = 'snow://notebook/DEV_SUPPORT.DBT.DBT_GENERATOR_NEGOCIO/versions/live'  # copia exacta desde Files > Copy path
#snow://notebook/DEV_SUPPORT.DBT.DBT_GENERATOR_NEGOCIO/versions/live/DBT_GENERATOR_NEGOCIO.ipynb
P = {
    "MODEL_TYPE": "MODEL_TYPE",
    "INEG_TYPE": "INEG_TYPE",
    "BBDD_SRC_NAME": "BBDD_SRC_NAME",
    "SCHM_SRC_NAME": "SCHM_SRC_NAME",
    "TABLE_SRC_NAME": "TABLE_SRC_NAME",
    "COLUMN_SRC_ID": "COLUMN_SRC_ID",
    "COLUMN_SRC_NAME": "COLUMN_SRC_NAME",
    "BBDD_TGT_NAME": "BBDD_TGT_NAME",
    "SCHM_TGT_NAME": "SCHM_TGT_NAME",
    "TABLE_TGT_NAME": "TABLE_TGT_NAME",
    "COLUMN_TGT_NAME": "COLUMN_TGT_NAME",
    "COLUMN_FORM": "COLUMN_FORM",
    "COLUMN_PK": "COLUMN_PK",
    "COLUMN_COALESCE": "COLUMN_COALESCE",
    "COLUMN_COALESCE_DEFAULT": "COLUMN_COALESCE_DEFAULT",
    "COLUMN_NOT_NULL": "COLUMN_NOT_NULL",
    "COLUMN_HASH": "COLUMN_HASH",
    "COLUMN_EXP": "COLUMN_EXP",
    "COLUMN_STD": "COLUMN_STD",
    "COLUMN_EQUIVAL": "COLUMN_EQUIVAL",
    "COLUMN_EQUIVAL_ROLE": "COLUMN_EQUIVAL_ROLE",
    "COLUMN_TAG": "COLUMN_TAG",
    "COLUMN_ENCRYPT": "COLUMN_ENCRYPT"
}

A = {
    "tec_des_cod_siglas": "tec_des_cod_siglas",
    "tec_des_empresa": "tec_des_empresa",
    "tec_id_ingesta": "tec_id_ingesta",
    "tec_ts_ingesta": "tec_ts_ingesta",
    "tec_ts_staging": "tec_ts_staging",
    "tec_ts_integracion_b": "tec_ts_integracion_b",
    "tec_fec_inicio": "tec_fec_inicio",
    "tec_fec_fin": "tec_fec_fin",
    "tec_cod_vigencia": "tec_cod_vigencia",
    "tec_ts_integracion_s": "tec_ts_integracion_s",
    "tec_hash": "tec_hash",
    "timestamp": "timestamp",
}

COD_SIGLAS = {
    "DELISANO": "DLS",
    "COLLELL": "COL",
    "COOPECARN": "COO",
    "FCB": "FCB",
    "FRICAFOR": "FRI",
    "MONTER": "MON",
    "CBMF": "CBM",
    "GC": "GRC",
    "LATAM": "LAT"
}

In [ ]:
df = pd.read_excel('DBT_GENERATOR/param_table_negocio.xlsx', sheet_name='Param_Table')
df = df.sort_values([P['BBDD_SRC_NAME'], P['SCHM_SRC_NAME'], P['TABLE_SRC_NAME'], P['COLUMN_SRC_ID']], na_position='last', kind='mergesort').fillna('')

def split_multi(val: str) -> list:
    """Divide una lista en texto por coma/semicolon y limpia espacios."""
    if pd.isna(val): return []
    return [x.strip() for x in re.split(r'[;,]', str(val)) if x.strip()]

def normalize_schema_lists(df: pd.DataFrame, schm_col: str = "SCHM_SRC_NAME") -> pd.DataFrame:
    """
    Devuelve un DF 'explosionado' con una fila por esquema en SCHM_SRC_NAME.
    Soporta tanto el v3 (un esquema) como el v4 (lista de esquemas por coma).
    """
    rows = []
    for _, r in df.iterrows():
        schms = split_multi(r.get(schm_col, "")) or [r.get(schm_col, "")]
        #schms = r.get(schm_col, "")
        for schm in schms:
            rr = r.copy()
            rr[schm_col] = schm
            rows.append(rr)
    return pd.DataFrame(rows)
    

df = normalize_schema_lists(df, 'SCHM_SRC_NAME')
df = df.sort_values([P['BBDD_SRC_NAME'], P['SCHM_SRC_NAME'], P['TABLE_SRC_NAME'], P['COLUMN_SRC_ID']], na_position='last', kind='mergesort').fillna('').reset_index(drop=True)

print(df.shape)
df

In [ ]:
# --- ESTRUCTURA DE CARPETAS ---
asociados = extract_asociados_from_bronze(df, 'BBDD_SRC_NAME', 'SCHM_SRC_NAME', 'DEV_BRONZE', 'BRONZE_')

dbt_project_yml = textwrap.dedent(f"""
  name: CBMF_DBT
  version: 1.0.0
  config-version: 2
  profile: CBMF_DBT
  model-paths:
    - models
  analysis-paths:
    - analyses
  test-paths:
    - tests
  seed-paths:
    - seeds
  macro-paths:
    - macros
  snapshot-paths:
    - snapshots
  models:
    CBMF_DBT:
      +materialized: view
  vars:
    empresas: {yml_list_str(asociados)}
    dbt_test_audit_database: "{{{{ 'PROD_SUPPORT' if target.name == 'prod' else 'DEV_SUPPORT' }}}}"
    dbt_test_audit_schema: DBT
  on-run-start:
    - "{{{{ audit__ensure_test_audit_objects() }}}}"
  on-run-end:
    - "{{{{ audit__log_test_results(results) }}}}"
""").strip()


# --- profiles.yml ---
profiles_yml = textwrap.dedent("""
  CBMF_DBT:
    target: dev
    outputs:
      dev:
        type: snowflake
        role: DEV_ETL
        warehouse: WH_SILVER_DEV
        database: DEV_SUPPORT
        schema: DBT
        account: ' '
        user: ' '
      prod:
        type: snowflake
        role: PROD_ETL
        warehouse: WH_SILVER_PROD
        database: PROD_SUPPORT
        schema: DBT
        account: ' '
        user: ' '
""").strip()


# --- .gitignore ---
gitignore = textwrap.dedent("""
  # Ignore dbt artifacts
  target/
  dbt_modules/
  logs/
""").strip()


stc_cast_sql = textwrap.dedent("""
  {% macro std_cast(expr, dtype, fmt=None, scale=None) %}
      {# 1) Prepara la expresión base, limpia espacios y convierte '' a NULL #}
      {%- set e = "NULLIF(TRIM(" ~ expr ~ "), '')" -%}

      {# 2) Normaliza el tipo a mayúsculas #}
      {%- set t = (dtype or 'VARCHAR') | upper -%}

      {# 3) Ramas por tipo destino: #}

      {%- if t in ["TEXT","STRING","VARCHAR"] %}
          {{ e }}  {# devuelve texto limpio o NULL si '' #}

      {%- elif t in ["INT","INTEGER"] %}
          (TRY_TO_NUMBER({{ e }}))::NUMBER(38, 0)

      {%- elif t in ["NUMBER","DECIMAL","NUMERIC"] %}
          {%- if scale is not none %}
              (TRY_TO_NUMBER({{ e }}, 38, {{ scale }}))::NUMBER(38,{{ scale }})
          {%- else %}
              (TRY_TO_NUMBER({{ e }}))::NUMBER(38, 0)
          {%- endif %}

      {%- elif t in ["DATE"] %}
          {%- if fmt %}
              TRY_TO_DATE({{ e }}, '{{ fmt }}')
          {%- else %}
              TRY_TO_DATE({{ e }})
          {%- endif %}

      {%- elif t in ["TIMESTAMP_NTZ","TIMESTAMP"] %}
          COALESCE(TRY_TO_TIMESTAMP_NTZ({{ e }}),TO_TIMESTAMP_NTZ('1900-01-01 00:00:00'))

      {%- elif t in ["BOOLEAN","BOOL"] %}
          {# Para booleanos usamos el expr “en crudo” con TRIM/UPPER y mapeamos a TRUE/FALSE/NULL #}
          IFF(UPPER(TRIM({{ expr }})) IN ('TRUE','T','1','S','SI','Y','YES'), TRUE,
              IFF(UPPER(TRIM({{ expr }})) IN ('FALSE','F','0','N','NO'), FALSE, NULL))::BOOLEAN

      {%- else %}
          {{ expr }}  {# fallback si aparece un tipo no contemplado #}
      {%- endif %}
  {% endmacro %}
""").strip()


generate_schema_name = textwrap.dedent("""
  {% macro generate_schema_name(custom_schema_name, node) -%}
    {%- if custom_schema_name is none -%}
      {{ target.schema }}
    {%- else -%}
      {{ custom_schema_name }}
    {%- endif -%}
  {%- endmacro %}
""")

hard_delete = textwrap.dedent("""
{% macro hard_delete_against_src(this_model, src_model, keys) %}
delete from {{ this_model }} t
where not exists (
  select 1
  from {{ src_model }} s
  where 
    {% for k in keys %}
    s.{{ k }} is not distinct from t.{{ k }}{% if not loop.last %} and {% endif %}
    {% endfor %}
);
{% endmacro %}
""")

equiv_or_default = textwrap.dedent("""
{% macro join_cfg_equivalencias(tabla, variable, emp_col, codigo_col) %}
left join (
    select * from {{ source('bronze_comun','CFG_EQUIVALENCIAS') }}
    where tabla    = '{{ tabla }}'
      and variable = '{{ variable }}'
      and coalesce(f_inicio, date('1900-01-01')) <= current_date()
      and coalesce(f_final,  date('2999-12-31')) >= current_date()
      and emp in ('{{ emp_col }}', 'ALL')
    qualify row_number() over (
        partition by tabla, variable, codigo
        order by
            case when emp = '{{ emp_col }}' then 0 else 1 end,
            coalesce(f_inicio, date('1900-01-01')) desc
    ) = 1
) eq
  on eq.codigo = {{ codigo_col }}
{% endmacro %}
""")


audit_test_results = textwrap.dedent("""
{# =========================
   Auditoría de resultados de dbt tests en una única tabla
   ========================= #}
{# Solo se lanza cuando se está ejecutando un test (test / build) #}
{% macro audit__should_log_tests() %}
  {{ return(flags.WHICH in ['test', 'build']) }} {# flags.WHICH para sacar el comando ejecutado (run/test/build/compile/...) #}
{% endmacro %}


{# Crea la tabla en DDBB y schema especificado en las variables en project.yml, si no se define ninguna, usa target.database y target.schema_AUDIT #}
{% macro audit__audit_relation() %}
  {% set audit_db = var('dbt_test_audit_database', target.database) %}
  {% set audit_schema = var('dbt_test_audit_schema', target.schema ~ '_AUDIT') %}
  {% set audit_table = var('dbt_test_audit_table', 'DBT_TEST_RESULTS') %}
  {{ return(api.Relation.create(database=audit_db, schema=audit_schema, identifier=audit_table)) }} {# Referenciamos mediante relation para que dbt gestione quoting correctamente #}
{% endmacro %}


{# Convierte un valor Jinja a literal SQL seguro (escapa comillas) o NULL #}
{% macro audit__sql_literal(val) %}
  {% if val is none %}
    null
  {% else %}
    '{{ (val | string) | replace("'", "''") }}'
  {% endif %}
{% endmacro %}


{% macro audit__ensure_test_audit_objects() %}
  {% if not audit__should_log_tests() %} {# Scape si no estamos en test/build #}
    {{ return('select 1') }}
  {% endif %}
  {% set rel = audit__audit_relation() %} {# Identificamos la tabla destino #}
  {% if execute %} {# execute=True solo cuando dbt está ejecutando (no en parse). run_query requiere conexión #}
    {% set ddl %} {# Creamos tabla de auditoría si no existe #}
      create table if not exists {{ rel }} (
          inserted_at           timestamp_tz default current_timestamp(), {# TS de inserción #}
          run_started_at        timestamp_tz, {# TS de inicio de esa ejecución #}
          target_name           string, {# prod/dev #}
          target_database       string, {# DDBB #}
          target_schema         string, {# Schema #}
          original_file_path    string, {# Path del fichero #}
          invocation_id         string, {# ID de ejecución #}
          node_unique_id        string, {# ID de test #}
          node_name             string, {# Nombre del modelo test (modelo+columna+test) #}
          test_name             string, {# Nombre del test #}
          depends_nodes         variant, {# De que modelos depende #}
          column_name           string, {# De que campos depende #}
          test_kwargs           variant, {# Argumentos del test #}
          status                string, {# pass, fail, warn, error, skipped #}
          failures              number, {# Nº de fallos #}
          failure_details       variant,
          execution_time_s      float, {# Tiempo de ejecución (s) #}
          message               string {# Mensaje del resultado #}
      )
    {% endset %}
    {% do run_query(ddl) %}
  {% endif %}
  {{ return('select 1') }} {# El hook necesita devolver algo #}
{% endmacro %}


{% macro audit__log_test_results(results) %}
  {% if not audit__should_log_tests() %} {# Si no estamos en test/build, no hacemos nada #}
    {{ return('select 1') }}
  {% endif %}
  {% set rel = audit__audit_relation() %} {# Tabla destino #}
  {% if execute %} {# Solamente si ejecutamos #}
    {% set rows = [] %} {# Construimos un INSERT masivo (UNION ALL) con solo los Result cuyo resource_type == 'test' #}
    {% for res in results %}
      {% if res.node.resource_type == 'test' %}
        {% set test_name = (res.node.test_metadata.name if res.node.test_metadata is not none else none) %}
        {% set depends_nodes = (res.node.depends_on.nodes if res.node.depends_on is not none else []) %}
        {% set test_kwargs = (res.node.test_metadata.kwargs if res.node.test_metadata is not none else {}) %}
        
        {% set failure_details_sql = "null" %}
        {% if test_name == 'no_nulls_except' and res.status in ['fail','warn'] and res.node.compiled_code is not none %}
          {% set compiled = res.node.compiled_code %}
          {% if compiled.endswith(';') %} {# quitar ; final si existe #}
            {% set compiled = compiled[:-1] %}
          {% endif %}
          {% set failure_details_sql %}
            (
              select array_agg(
                       object_construct(
                         'column_name', column_name::string,
                         'null_count', null_count::number
                       )
                     )
              from ( {{ compiled }} ) t
            )
          {% endset %}
        {% endif %}


        {% set row_sql %}
          select
            {{ audit__sql_literal(run_started_at) }}::timestamp_tz                  as run_started_at,
            {{ audit__sql_literal(target.name) }}                                   as target_name,
            {{ audit__sql_literal(target.database) }}                               as target_database,
            {{ audit__sql_literal(target.schema) }}                                 as target_schema,
            {{ audit__sql_literal(res.node.original_file_path) }}                   as original_file_path,
            {{ audit__sql_literal(invocation_id) }}                                 as invocation_id,
            {{ audit__sql_literal(res.node.unique_id) }}                            as node_unique_id,
            {{ audit__sql_literal(res.node.name) }}                                 as node_name,
            {{ audit__sql_literal(test_name) }}                                     as test_name,
            parse_json({{ audit__sql_literal(tojson(depends_nodes)) }})             as depends_nodes,
            {{ audit__sql_literal(res.node.column_name) }}                          as column_name,
            parse_json({{ audit__sql_literal(tojson(test_kwargs)) }})               as test_kwargs,
            {{ audit__sql_literal(res.status) }}                                    as status,
            {{ audit__sql_literal(res.failures) }}                                  as failures,
            {{ failure_details_sql }}                                               as failure_details,
            {{ res.execution_time }}                                                as execution_time_s,
            {{ audit__sql_literal(res.message) }}                                   as message
        {% endset %}
        {% do rows.append(row_sql) %}
      {% endif %}
    {% endfor %}
    {% if rows | length > 0 %} {# Insertamos solo si hay filas (si no, evitamos SQL inválido) #}
      {% set insert_sql %}
        insert into {{ rel }} (
          run_started_at, target_name, target_database, target_schema, original_file_path, invocation_id, node_unique_id, node_name, test_name,
          depends_nodes, column_name, test_kwargs, status, failures, failure_details, execution_time_s, message
        )
        {{ rows | join("\\nunion all\\n") }}
      {% endset %}
      {% do run_query(insert_sql) %}
    {% endif %}
  {% endif %}
  {{ return('select 1') }}
{% endmacro %}
""")

incremental_merge_identity = textwrap.dedent("""
{% materialization incremental_merge_identity, adapter='snowflake' %}

  {%- set target_relation = this -%}

  {# 0) La tabla debe existir (porque tiene IDENTITY) #}
  {%- set existing_relation = load_cached_relation(target_relation) -%}
  {%- if existing_relation is none -%} {# Exception #}
    {{ exceptions.raise_compiler_error(
      "incremental_merge_identity requiere que la tabla destino exista previamente (para preservar IDENTITY). " ~
      "No existe: " ~ target_relation
    ) }}
  {%- endif -%}

  {# 0. Asignamos campos de configuración del modelo #}
  {%- set unique_key = config.get('unique_key') -%} {# Unique key #}
  {%- if unique_key is none -%} {# Exception #}
    {{ exceptions.raise_compiler_error("Falta config 'unique_key' en incremental_merge_identity.") }}
  {%- endif -%}
  {%- if unique_key is not string -%} {# Normaliza solo si es lista #}
    {%- set unique_key = unique_key | list -%}
  {%- endif -%}
  {%- set merge_exclude = (config.get('merge_exclude_columns', []) | map('upper') | list) -%} {# merge_exclude #}
  {%- set identity_cols = (config.get('identity_columns', []) | map('upper') | list) -%} {# identity_cols #}

  {# 1. Creamos staging como tabla temporal con el SQL del modelo #}
  {%- set tmp_relation = make_temp_relation(target_relation) -%} {# Objeto relation que representa una tabla temporal donde volcamos el SELECT del modelo #}
  {% do run_query(create_table_as(True, tmp_relation, sql)) %} {# create temporary table <tmp_relation> as <sql>; True = temporal#}

  {# 2. Identificamos columnas destino y staging #}
  {%- set dest_cols = adapter.get_columns_in_relation(target_relation) -%}
  {%- set src_cols  = adapter.get_columns_in_relation(tmp_relation) -%}
  {%- set src_names = src_cols | map(attribute='name') | map('upper') | list -%} {# Para poder mapear en step 3 (target ∩ source) cols #}

  {# 3. insert_cols = (target ∩ source) - identity #}
  {%- set insert_cols = [] -%}
  {%- for c in dest_cols -%}
    {%- set cn = c.name | upper -%}
    {%- if cn in identity_cols -%}{% continue %}{%- endif -%}
    {%- if cn in src_names -%}{% do insert_cols.append(c.name) %}{%- endif -%}
  {%- endfor -%}
  {%- if insert_cols | length == 0 -%} {# Exception #}
    {{ exceptions.raise_compiler_error("incremental_merge_identity: no hay columnas para insertar (insert_cols vacío).") }}
  {%- endif -%}

  {# 4. update_cols = (target ∩ source) - identity - merge_exclude #}
  {%- set update_cols = [] -%}
  {%- for c in dest_cols -%}
    {%- set cn = c.name | upper -%}
    {%- if cn in identity_cols -%}{% continue %}{%- endif -%}
    {%- if cn in merge_exclude -%}{% continue %}{%- endif -%}
    {%- if cn in src_names -%}{% do update_cols.append(c.name) %}{%- endif -%}
  {%- endfor -%}

  {# 5. ON clause: unique_key string o lista #}
  {%- if unique_key is string -%}
    {%- set on_clause = "t." ~ unique_key ~ " is not distinct from s." ~ unique_key -%}
  {%- else -%}
    {%- set preds = [] -%}
    {%- for k in unique_key -%}
      {%- do preds.append("t." ~ k ~ " is not distinct from s." ~ k) -%}
    {%- endfor -%}
    {%- set on_clause = preds | join(" and ") -%}
  {%- endif -%}

  {# 6. VALUES: s.col1, s.col2, ... #}
  {%- set insert_values = [] -%}
  {%- for c in insert_cols -%}
    {%- do insert_values.append("s." ~ c) -%}
  {%- endfor -%}

  {# 7. UPDATE SET: t.col = s.col, ... #}
  {%- set update_setters = [] -%}
  {%- for c in update_cols -%}
    {%- do update_setters.append("t." ~ c ~ " = s." ~ c) -%}
  {%- endfor -%}

  {# 8. SQL del MERGE #}
  {% set merge_sql %}
    merge into {{ target_relation }} t
    using {{ tmp_relation }} s
      on {{ on_clause }}
    when matched then update set
      {{ update_setters | join(",\\n      ") }}
    when not matched then insert ({{ insert_cols | join(", ") }})
    values ({{ insert_values | join(", ") }})
  {% endset %}

  {# 9. Ejecutar el SQL principal dentro de statement('main') #}
  {% call statement('main') %}
    {{ merge_sql }}
  {% endcall %}

  {# 10. Limpieza staging #}
  {% do run_query("drop table if exists " ~ tmp_relation) %}

  {{ return({'relations': [target_relation]}) }}

{% endmaterialization %}
""")


write_text("dbt_project.yml", dbt_project_yml)
write_text("profiles.yml", profiles_yml)
write_text(".gitignore", gitignore)
for d in ["analyses", "macros", "models", "seeds", "snapshots", "tests"]:
    write_text(f"{d}/.gitkeep", "")   # fuerza la carpeta
write_text("macros/std_cast.sql", stc_cast_sql)
write_text("macros/generate_schema_name.sql", generate_schema_name)
write_text("macros/hard_delete.sql", hard_delete)
write_text("macros/equiv_or_default.sql", equiv_or_default)
write_text("macros/audit_test_results.sql", audit_test_results)
write_text("macros/incremental_merge_identity.sql", incremental_merge_identity)

In [ ]:
no_nulls_except = textwrap.dedent("""
{% test no_nulls_except(model, exclude=[]) %}
{# Falla si hay al menos un NULL en cualquier columna del modelo, excepto las listadas en exclude. Devuelve column_name y null_count. #}

{% set cols = adapter.get_columns_in_relation(model) %}
{% set exu  = exclude | map('upper') | list %}

{# columnas a testear (sin las excluidas) #}
{% set test_cols = [] %}
{% for c in cols %}
  {% if c.name | upper not in exu %}
    {% do test_cols.append(c.name) %}
  {% endif %}
{% endfor %}

{% if test_cols | length == 0 %}
  -- Nada que testear: pasa
  select 0 where 1=0
{% else %}
  with agg as (
    select object_construct(
      {%- for c in test_cols -%}
        '{{ c }}', sum(case when {{ c }} is null then 1 else 0 end)
        {%- if not loop.last %}, {% endif -%}
      {%- endfor -%}
    ) as null_counts
    from {{ model }}
  ),
  flat as (
    select
      f.key::string   as column_name,
      f.value::number as null_count
    from agg, lateral flatten(input => agg.null_counts) f
  )
  select column_name, null_count
  from flat
  where null_count > 0
{% endif %}
{% endtest %}


----------------------------------------------------------------------------------------------------

""")

single_batch = textwrap.dedent(f"""
{{% test single_batch(model, column_name='{A['tec_ts_ingesta']}') %}}
-- Falla si hay >1 batch en la view
select 1
from {{{{ model }}}}
having count(distinct {{{{ column_name }}}}) > 1
{{% endtest %}}

----------------------------------------------------------------------------------------------------

""")

no_stale_batch_vs_integ = textwrap.dedent(f"""
{{% test no_stale_batch_vs_integ(model, integ_ref, pk_cols, assoc_col='{A['tec_des_empresa']}', integ_ts_col='{A['tec_ts_ingesta']}') %}}

{{# Construimos las condiciones de join por PK: b.col = i.col #}}
{{% set join_conds = [] %}}
{{% for col in pk_cols %}}
  {{% set _ = join_conds.append('b.' ~ col ~ ' = i.' ~ col) %}}
{{% endfor %}}
{{% set pk_join_sql = join_conds | join(' AND ') %}}

{{# SELECT de PKs para el GROUP BY #}}
{{% set pk_select = pk_cols | join(', ') %}}

with integ_max as (
  select {{{{ pk_select }}}}, {{{{ assoc_col }}}} as {{{{ assoc_col }}}}, max({{{{ integ_ts_col }}}}) as max_integ_ts
  from {{{{ ref(integ_ref) }}}}
  where {A['tec_cod_vigencia']} > 0
  group by {{{{ pk_select }}}}, {{{{ assoc_col }}}}
)
select b.*
from {{{{ model }}}} b
join integ_max i
  on b.{{{{ assoc_col }}}} = i.{{{{ assoc_col }}}}
 and {{{{ pk_join_sql }}}}
where i.max_integ_ts > b.{A['tec_ts_ingesta']}
{{% endtest %}}

----------------------------------------------------------------------------------------------------

""")

scd2_no_overlap = textwrap.dedent(f"""
{{% test scd2_no_overlap(model, pk_cols, start_col='{A['tec_fec_inicio']}', assoc_col='{A['tec_des_empresa']}', end_col='{A['tec_fec_fin']}') %}}
{{% set pk_select = (pk_cols | map('trim') | list) | join(', ') %}}
with ord as (
  select
    {{{{ pk_select }}}},
    {{{{ start_col }}}} as start_dt,
    {{{{ end_col }}}}   as end_dt,
    lead({{{{ start_col }}}}) over (partition by {{{{ pk_select }}}}, {{{{ assoc_col }}}} order by {{{{ start_col }}}}) as next_start
  from {{{{ model }}}}
),
bad as (
  select 1
  from ord
  where next_start is not null
    and next_start <= end_dt
)
select * from bad
{{% endtest %}}

----------------------------------------------------------------------------------------------------

""")

scd2_single_current = textwrap.dedent(f"""
{{% test scd2_single_current(model, pk_cols, assoc_col='{A['tec_des_empresa']}', current_col='{A['tec_cod_vigencia']}') %}}
{{% set pk_select = (pk_cols | map('trim') | list) | join(', ') %}}
with g as (
  select {{{{ pk_select }}}}, {{{{ assoc_col }}}},
         sum(case when {{{{ current_col }}}} > 0 then 1 else 0 end) as curr_cnt
  from {{{{ model }}}}
  group by {{{{ pk_select }}}}, {{{{ assoc_col }}}}
)
select 1 from g where curr_cnt <> 1
{{% endtest %}}

----------------------------------------------------------------------------------------------------

""")

generic_tests= no_nulls_except + single_batch + no_stale_batch_vs_integ + scd2_no_overlap + scd2_single_current

write_text("macros/tests/generic_tests.sql", generic_tests)

In [ ]:
def write_sources_file(
    df: pd.DataFrame,
    v_model_type: str = "INTEGRATION",
    rel_path: str = "models/sources.yml",
) -> None:
    """
    Genera un sources.yml a partir de la parametría.
    - Agrupa por (BBDD_SRC_NAME, SCHM_SRC_NAME)
    - Lista tablas únicas por cada (db, schema)
    - Escribe name: <schema_lower> para que coincida con source('<schema_lower>', '<tabla>')
    """
    # 1) filtro
    if P['MODEL_TYPE'] not in df.columns:
        raise ValueError(f"Falta columna {P['MODEL_TYPE']}.")
    sdf = df[df[P['MODEL_TYPE']].astype(str).str.upper() == str(v_model_type).upper()].copy()
    if sdf.empty:
        return

    # 2) claves
    for c in (P['BBDD_SRC_NAME'], P['SCHM_SRC_NAME'], P['TABLE_SRC_NAME']):
        if c not in sdf.columns:
            raise ValueError(f"Falta columna {c}.")
    schemas = sdf[[P['BBDD_SRC_NAME'], P['SCHM_SRC_NAME']]].drop_duplicates()

    # 3) YAML
    lines = []
    lines.append("version: 2\n")
    lines.append("sources:")
    for _, row in schemas.iterrows():
        db = str(row[P['BBDD_SRC_NAME']])
        schm = str(row[P['SCHM_SRC_NAME']])
        src_name = schm.lower()
        dev_db, prod_db = db_to_env_dbs(db)
        
        lines.append(f"  - name: {src_name}")
        #lines.append(f"    database: {db}")
        lines.append(f'''    database: "{{{{ '{prod_db}' if target.name == 'prod' else '{dev_db}' }}}}"''')
        lines.append(f"    schema: {schm}")
        lines.append(f"    quoting:")
        lines.append(f"      database: false")
        lines.append(f"      schema: false")
        lines.append(f"      identifier: true")
        lines.append(f"    tables:")

        tables = (sdf[(sdf[P['BBDD_SRC_NAME']] == db) & (sdf[P['SCHM_SRC_NAME']] == schm)][P['TABLE_SRC_NAME']]
                  .drop_duplicates().tolist())
        for t in tables:
            lines.append(f"      - name: {t}")
            lines.append(f"        description: 'Tabla de ingesta de datos {db}.{schm}.{t}'")
            lines.append(f"        loaded_at_field: {A['tec_ts_staging']}")

    lines.append("""
  - name: bronze_comun
    database: "{{ 'PROD_BRONZE' if target.name == 'prod' else 'DEV_BRONZE' }}"
    schema: BRONZE_COMUN
    quoting:
      database: false
      schema: false
      identifier: true
    tables:
      - name: ISO_3166_COUNTRY_CODES
        description: 'Seed de geografía y países para D_GEOGRAFICA y D_PAIS'
      - name: CFG_LITERALES
        description: 'Seed para carga de tablas extra'
      - name: CFG_EQUIVALENCIAS
        description: 'Seed equivalencias'
      - name: CFG_ETL_PARAMS
        description: 'Seed parametría'
      - name: CA_POB
        description: 'Seed poblaciones y comunidades autónomas'

  - name: support_param
    database: "{{ 'PROD_SUPPORT' if target.name == 'prod' else 'DEV_SUPPORT' }}"
    schema: SUPPORT_PARAM
    quoting:
      database: false
      schema: false
      identifier: true
    tables:
      - name: INSERT_LOGS
        description: 'Log de ingestas ADF para batches BRONZE'

  - name: gold_comun
    database: "{{ 'PROD_GOLD' if target.name == 'prod' else 'DEV_GOLD' }}"
    schema: GOLD_COMUN
    quoting:
      database: false
      schema: false
      identifier: true
    tables:
      - name: D_CALENDARIO_M
        description: 'Calendario mensual común'""")

    content = "\n".join(lines) + "\n"

    write_text(rel_path, content)

write_sources_file(df, rel_path="models/sources.yml")


In [ ]:

def generate_granuled_batch_views(
    df: pd.DataFrame,
    models_subdir: str = "models/bronze",
    extract_string_part_starts_tbl_src: str = "V_ETL_", extract_string_part_starts_schm_src: str = "BRONZE_", extract_string_part_ends_schm_src: str = "_P",
    v_model_type: str = "INTEGRATION"
):
    
    # 1. Nos quedamos con filas de integración
    dfi = df[df[P['MODEL_TYPE']].str.upper().eq(v_model_type.upper())].copy()
    if dfi.empty:
        print("No hay filas INTEGRATION en la parametría.")
        return

    # 2. Agrupamos por dataset de origen → 1 vista por (DB, schema, tabla) = asociado/entidad y definimos path
    grp = dfi.groupby([P['BBDD_SRC_NAME'], P['SCHM_SRC_NAME'], P['TABLE_SRC_NAME']], dropna=False)
    root_rel = f"{models_subdir}".strip("/")

    # 3. Iteramos por cada grupo para generar la vista correspondiente
    for (db_src, schm_src, tbl_src), g in grp:
        # Storeamos valores clave
        entidad   = extract_string_part(tbl_src, starts=extract_string_part_starts_tbl_src)
        # Nombre canónico Silver desde TABLE_TGT_NAME (puede diferir del src por empresa)
        tbl_tgt_raw = str(g[P['TABLE_TGT_NAME']].dropna().iloc[0]).strip() if not g[P['TABLE_TGT_NAME']].dropna().empty else tbl_src
        entidad_tgt = extract_string_part(tbl_tgt_raw)
        asociado  = extract_string_part(schm_src, starts=extract_string_part_starts_schm_src, ends = extract_string_part_ends_schm_src).upper()
        asociado_cuoted = f"'{asociado}'"
        source_nm = (schm_src or "").lower()
        table_nm  = (tbl_src or "").upper()

        # Equivalencias
        has_equiv = False
        equiv_var = None        # COLUMN_TGT_NAME del campo equivalenciado (ej. COD_BAJA)
        equiv_expr = None       # expresión de código (ej. "Cod_ baja")
        used_des    = False

        # Definimos path de salida (usando nombre canónico target)
        out_file = f"{root_rel}/{entidad_tgt}_batch_{asociado.lower()}.sql"

        # Ordenamos columnas
        if P['COLUMN_SRC_ID'] in g.columns:
            g_ord = g.sort_values(P['COLUMN_SRC_ID'], kind="mergesort")
        else:
            g_ord = g.sort_values(P['COLUMN_SRC_NAME'], kind="mergesort")

        # lista SELECT: usa COLUMN_EXP si viene (raw), si no, la columna origen quoted
        # si COLUMN_STD = TRUE, aplicamos macro std_cast(expr, dtype [,fmt] [,scale])
        select_lines = []
        tgt_names_lower = set()
        # mapa para poder construir filtros NOT NULL sobre CAMPOS NOT NULL usando la expresión real
        expr_by_tgt_upper = {}
        dtype_by_tgt_upper = {}

        for _, row in g_ord.iterrows():
            tgt_raw = (row.get(P['COLUMN_TGT_NAME']) or "").strip()
            if not tgt_raw:
                tgt_raw = (row.get(P['COLUMN_SRC_NAME']) or "").strip()
                if not tgt_raw:
                    continue
            tgt = quoted(tgt_raw)

            # 1) expresión base
            expr = (row.get(P['COLUMN_EXP']) or "").strip()
            if not expr:
                src_col = (row.get(P['COLUMN_SRC_NAME']) or "").strip()
                expr = quoted_src_cls(src_col) if src_col else "NULL"

            # FLAGS
            std_flag = parse_bool(row.get(P['COLUMN_STD'], False)) # STD flag
            role_raw = (row.get(P.get('COLUMN_EQUIVAL_ROLE','')) or '').strip().upper()
            equiv_role = role_raw or ('COD' if parse_bool(row.get(P['COLUMN_EQUIVAL'], False)) else '')
            column_encrypt = parse_bool(row.get(P['COLUMN_ENCRYPT'], False))
            coalesce_flag = parse_bool(row.get(P['COLUMN_COALESCE'], False))

            dtype = (row.get(P['COLUMN_FORM']) or "VARCHAR").strip().upper()

            default_col = P.get('COLUMN_COALESCE_DEFAULT')
            default_raw = row.get(default_col) if default_col else None
            default_lit_override = normalize_default_literal(default_raw, dtype) if default_raw is not None else None
            
            if std_flag and expr != "NULL":
                # construimos la llamada al macro con la expr entrecomillada para Jinja
                macro_args = [jinja_sql_str(expr), f"'{dtype}'"]
                macro_call = "{{ std_cast(" + ", ".join(macro_args) + ") }}"
                base_expr = macro_call
                base_expr0 = macro_call
            else:
                base_expr = expr
                base_expr0 = expr

            # Si marcamos más de una columna como equivalenciada, de momento no lo soportamos
            if equiv_role == 'COD':
                if has_equiv:
                    raise ValueError(f"Más de un COD en equivalencias: {schm_src}/{tbl_src}")
                has_equiv = True
                equiv_var = tgt_raw.strip().upper() # ej. COD_BAJA
                equiv_expr = expr #base_expr0  # ej. "\"Cod_ baja\""
                base_expr = f"coalesce(eq.codigo_eq, {base_expr})"
            if equiv_role == 'DES':
                if used_des:
                    raise ValueError(f"Más de una descripción en equivalencias: {schm_src}/{tbl_src}")
                used_des = True
                base_expr = f"coalesce(eq.descripcion, {base_expr})"

            # Guardamos expresión/dtype por target para luego construir filtros PK NOT NULL
            tgt_key = tgt_raw.strip().upper()
            expr_for_filters = base_expr # expr "real" para validaciones (NOT_NULL), PRE-coalesce
            expr_by_tgt_upper[tgt_key] = expr_for_filters
            dtype_by_tgt_upper[tgt_key] = dtype

            if coalesce_flag:  #Expr para el SELECT
                base_expr = apply_pk_coalesce(expr_for_filters, dtype, default_lit_override)
    
            if column_encrypt: base_expr = f"md5_hex({base_expr})"
            
            line = f"        {base_expr:<70} as {tgt}"
            select_lines.append(line)
            tgt_names_lower.add(tgt.lower())

        siglas = COD_SIGLAS.get(asociado)
        if not siglas:
            raise ValueError(f"Falta siglas para asociado={asociado}")

        # Añadimos auditoría mínima: sólo las columnas que NO vengan ya mapeadas desde la parametría.
        # Si la tabla fuente ya trae TEC_TS_INGESTA, TEC_DES_EMPRESA, etc. (cargadas por ADF),
        # no las volvemos a añadir para evitar "duplicate column" o "ambiguous column" en Snowflake.
        _audit_cols = [
            (f"        '{siglas}'{'':<68} as tec_des_cod_siglas", "tec_des_cod_siglas"),
            (f"        {asociado_cuoted:<70} as tec_des_empresa",  "tec_des_empresa"),
            ("        tec_id_ingesta",                             "tec_id_ingesta"),
            ("        tec_ts_ingesta",                             "tec_ts_ingesta"),
            ("        tec_ts_staging",                             "tec_ts_staging"),
            ("        tec_ts_integracion_b",                       "tec_ts_integracion_b"),
        ]
        for _line, _col in _audit_cols:
            if _col not in tgt_names_lower:
                select_lines.append(_line)

        # PKs para PARTITION BY
        pk_cols = [
            (row.get(P['COLUMN_TGT_NAME']) or "").strip()
            for _, row in g.iterrows()
            if parse_bool(row.get(P['COLUMN_PK'], False))
            and (row.get(P['COLUMN_TGT_NAME']) or "").strip()
        ]
        partition_by = ", ".join(pk_cols)

        # segundo criterio de ordenación: si existe 'secuencia', úsalo; si no, tec_ts_staging
        second_order = "TIMESTAMP DESC" if ("timestamp" in tgt_names_lower) else "tec_ts_staging DESC"

        qualify_clause = ""
        if partition_by:
            qualify_clause = textwrap.dedent(f"""
                QUALIFY ROW_NUMBER() OVER (
                    PARTITION BY {partition_by}
                    ORDER BY tec_ts_ingesta DESC, {second_order}
                ) = 1
            """).strip()

        # bloques listos para escribir
        #config_block = f"{{{{ config(database='{db_src}', schema='{schm_src}', materialized='view', tags=['bronze','batch',{asociado_cuoted}]) }}}}"
        dev_db, prod_db = db_to_env_dbs(db_src)
        db_expr = f"'{prod_db}' if target.name == 'prod' else '{dev_db}'"
        config_block = (
            f"{{{{ config(database={db_expr}, schema='{schm_src}', materialized='view', "
            f"tags=['bronze','batch',{asociado_cuoted}]) }}}}"
        )

        select_block = ",\n".join(select_lines)

        # Refs para source y logs
        src_ref_set = f"{{% set src_ref = source('{source_nm}', '{table_nm}') %}}"
        log_ref_set = "{% set log_ref = source('support_param', 'INSERT_LOGS') %}"

        buf = []
        buf.append(f"{config_block}")
        buf.append("")
        buf.append(src_ref_set)
        buf.append(log_ref_set)
        buf.append("")

        # next_log (batch pendiente más antiguo)
        buf.append("with next_log as (")
        buf.append("    select tec_id_ingesta")
        buf.append("    from {{ log_ref }}")
        buf.append("    where upper(database)   = upper('{{ src_ref.database }}')")
        buf.append("      and upper(schema)     = upper('{{ src_ref.schema }}')")
        buf.append("      and upper(table_name) = upper('{{ src_ref.identifier }}')")
        buf.append("      and start_watermark <> end_watermark")
        buf.append("      and tec_ts_integracion_b is null")
        buf.append("    qualify row_number() over(order by tec_ts_ingesta) = 1")
        buf.append("),")

        # src
        buf.append("src as (")
        buf.append("    select")
        buf.append(f"{select_block}")

        # JOIN equivalencias
        if used_des and not has_equiv:
            raise ValueError(f"En el proceso de equivalencias, hay DES sin COD en {schm_src}/{tbl_src}. Marca la pareja COD/DES correctamente.")
            
        if has_equiv and equiv_var and equiv_expr:
            buf.append("    from {{ src_ref }} s")

            # TABLA = entidad en mayúsculas
            tabla_equiv = entidad.strip().upper()   # ej. MOTIVO_BAJA
            variable_equiv = equiv_var             # ej. COD_BAJA
            emp_expr = asociado_cuoted             # ej. 'DELISANO'
    
            # Jinja dentro de f-string
            join_line = (
                "    {{ join_cfg_equivalencias("
                f"'{tabla_equiv}', '{variable_equiv}', {emp_expr}, 's.{equiv_expr}'"
                ") }}"
            )
            buf.append(join_line)
        else:
            buf.append("    from {{ src_ref }}")

        # WHERE por tec_id_ingesta del next_log
        buf.append("    where tec_id_ingesta = (select tec_id_ingesta from next_log)")

        # # Filtros NOT NULL - Recogemos los campos por los que debemos filtrar como NOT NULL
        not_null_cols = [
            ((row.get(P['COLUMN_TGT_NAME']) or row.get(P['COLUMN_SRC_NAME']) or "").strip().upper())
            for _, row in g.iterrows()
            if parse_bool(row.get(P['COLUMN_NOT_NULL'], False))
        ]
        not_null_cols = [c for c in not_null_cols if c]
                
        for c in not_null_cols:
            expr = expr_by_tgt_upper.get(c)
            datatype = (dtype_by_tgt_upper.get(c) or "").upper()
            dtype_base = datatype.split("(")[0].strip()
        
            if not expr:
                raise ValueError(f"COLUMN_NOT_NULL marca {c} pero no existe expr mapeada en {schm_src}/{tbl_src}")
        
            if dtype_base in ("VARCHAR","STRING","TEXT","CHAR","CHARACTER"):
                buf.append(f"      AND NULLIF(TRIM({expr}), '') IS NOT NULL")
            else:
                buf.append(f"      AND {expr} IS NOT NULL")

        buf.append(")")
        buf.append("select *")
        buf.append("from src")
            
        if qualify_clause:
            buf.append(f"{qualify_clause}\n")
        
        write_text(out_file, "\n".join(buf))
        print(f"[OK] {out_file}")


generate_granuled_batch_views(df)


In [ ]:
def generate_union_batch_views(
    df: pd.DataFrame,
    models_subdir: str = "models/silver",
    extract_string_part_starts_tbl_src: str = "V_ETL_",
    var_name: str = "empresas",
    v_model_type: str = "INTEGRATION",
) -> None:
    """
    Genera una vista de agregación por entidad (p.ej. cliente_batch.sql, pais_batch.sql) que hace UNION ALL
    de las vistas batch por asociado (cliente_batch_<asociado>, pais_batch_<asociado>, ...).
    """
    # 1) Filtramos INTEGRATION
    dfi = df[df[P['MODEL_TYPE']].astype(str).str.upper() == str(v_model_type).upper()].copy()
    if dfi.empty:
        print("No hay filas INTEGRATION en la parametría.")
        return

    # 2) Entidades únicas desde TABLE_TGT_NAME (nombre canónico Silver, igual en todas las empresas)
    dfi["__entidad__"] = dfi[P['TABLE_TGT_NAME']].astype(str).str.strip() \
        .apply(lambda x: extract_string_part(x))
    entidades = sorted(dfi["__entidad__"].dropna().unique().tolist())

    root_rel = f"{models_subdir}".strip("/")

    for entidad in entidades:
        if not entidad:
            continue

        # 3) Obtenemos (DB, SCHEMA) target únicos para la entidad
        sub = dfi[dfi["__entidad__"] == entidad]
        tgt_pairs = {
            (str(r.get(P['BBDD_TGT_NAME'], "")).strip(), str(r.get(P['SCHM_TGT_NAME'], "")).strip())
            for _, r in sub.iterrows()
            if str(r.get(P['BBDD_TGT_NAME'], "")).strip() and str(r.get(P['SCHM_TGT_NAME'], "")).strip()
        }
        if not tgt_pairs:
            print(f"[WARN] Sin DB/SCHEMA target para entidad '{entidad}'. Se omite.")
            continue

        # Si por error hay más de un (DB,SCHEMA), usamos el primero ordenado
        if len(tgt_pairs) > 1:
            msg = " / ".join([f"{db}.{sch}" for db, sch in sorted(tgt_pairs)])
            print(f"[WARN] Múltiples destinos para '{entidad}': {msg}. Usando el primero.")
        db_tgt, schm_tgt = sorted(tgt_pairs)[0]

        # Asociados que realmente tienen esta entidad (puede ser un subconjunto de todos)
        sub_asociados = sorted(
            sub[P['SCHM_SRC_NAME']].astype(str).str.strip()
            .apply(lambda x: extract_string_part(x, starts="BRONZE_"))
            .str.lower()
            .dropna().unique().tolist()
        )
        sub_asociados = [a for a in sub_asociados if a]
        jinja_list = "[" + ", ".join(f"'{a}'" for a in sub_asociados) + "]"

        # 4) Estructura y contenido del .sql
        out_dir  = root_rel
        out_file = f"{out_dir}/{entidad}_batch.sql"

        dev_db, prod_db = db_to_env_dbs(db_tgt)
        db_expr = f"'{prod_db}' if target.name == 'prod' else '{dev_db}'"
        content = textwrap.dedent(f"""
            {{{{ config(database={db_expr}, schema='{schm_tgt}', materialized='view', tags=['silver','batch']) }}}}

            {{%- set asociados = {jinja_list} -%}}

            {{% for a in asociados %}}
                {{% if not loop.first %}} union all {{% endif %}}
                select * from {{{{ ref('{entidad}_batch_' ~ a) }}}}
            {{% endfor %}}
        """).lstrip()

        write_text(out_file, content)
        print(f"[OK] {out_file}")

generate_union_batch_views(df)


In [ ]:
INTEG_EXTRA_WHERE_BY_ENTITY = {
    "transaccion_historica": "TIPO_CALCULO NOT IN (4, 8)"
}

def write_integration_models(
    df: pd.DataFrame,
    out_dir: str = "models/silver",
    starts_tbl_src: str = "V_ETL_", ends_tbl_src: str = "_TEST"
):
    """
    Genera los modelos SCD2 de integración por ENTIDAD en SILVER:
      - <entidad>_integ.sql
    """
    def uniq(seq):
        seen=set(); out=[]
        for x in seq:
            if x and x not in seen:
                seen.add(x); out.append(x)
        return out

    dfi = df[df[P['MODEL_TYPE']].astype(str).str.upper().eq("INTEGRATION")].copy()
    if dfi.empty:
        return []

    # ENTIDAD desde TABLE_TGT_NAME (nombre canónico Silver, igual para todas las empresas)
    dfi["__entidad__"] = dfi[P['TABLE_TGT_NAME']].astype(str).apply(lambda x: extract_string_part(x))

    outputs = []

    for entidad, dfe in dfi.groupby("__entidad__"):
        dfe = dfe.copy()
        entidad_ref = F'{entidad}'

        db_tgt   = dfe[P['BBDD_TGT_NAME']].dropna().iloc[0]
        dev_db, prod_db = db_to_env_dbs(db_tgt)
        db_expr = f"'{prod_db}' if target.name == 'prod' else '{dev_db}'"
        
        schm_tgt = dfe[P['SCHM_TGT_NAME']].dropna().iloc[0]
        schm_src   = str(dfe[P['SCHM_SRC_NAME']].dropna().iloc[0]).upper()
        src_suffix = " ~ '_p'" if schm_src.endswith("_P") else ""

        src_tbl  = dfe[P['TABLE_SRC_NAME']].dropna().iloc[0]
        tname    = f"{entidad}_integ".lower()

        # filtro extra (transacción h)
        extra_filter = INTEG_EXTRA_WHERE_BY_ENTITY.get(entidad_ref.lower(), "")
        src_where = f" where {extra_filter}" if extra_filter else ""

        # Nombre target
        dfe["__tgt_col__"] = dfe.apply(
            lambda r: (r.get(P['COLUMN_TGT_NAME']) or r.get(P['COLUMN_SRC_NAME'])),
            axis=1
        ).astype(str).str.strip()

        # Orden por COLUMN_SRC_ID si existe
        if P['COLUMN_SRC_ID'] in dfe.columns:
            dfe = dfe.sort_values(P['COLUMN_SRC_ID'])

        tgt_cols  = uniq([c for c in dfe["__tgt_col__"].tolist() if c])
        pk_cols   = uniq([c.upper() for c, flg in zip(dfe["__tgt_col__"], dfe[P['COLUMN_PK']])   if parse_bool(flg)])
        hash_cols = uniq([c.upper() for c, flg in zip(dfe["__tgt_col__"], dfe[P['COLUMN_HASH']]) if parse_bool(flg)])
        if hash_cols:
            hash_expr = "HASH(" + ", ".join(hash_cols) + ")"
        else:# Permitimos que tods los campos puedan ser PKs (ningún hash)
            hash_expr = "HASH('')"


        # SELECT base: todos los campos target + TIMESTAMP + auditoría + hash + AUD_START_DT (de timestamp)
        base_select = ", \n".join(
            ["        " + c for c in tgt_cols] + [
            "        tec_des_cod_siglas",
            "        tec_des_empresa",
            "        tec_id_ingesta",
            "        tec_ts_ingesta",
            f"        {hash_expr} AS tec_hash",
            "        date(tec_ts_ingesta) as tec_fec_inicio"
        ])

        # Condiciones por PK para pre-hook y cuerpo
        pk_join_update = " and ".join([f"t.{c} is not distinct from s.{c}" for c in pk_cols] + ["t.tec_des_empresa is not distinct from s.tec_des_empresa"]) if pk_cols else "1=1"
        pk_null_check  = f"t.{pk_cols[0]} is null" if pk_cols else "1=1"
        pk_list_prefix = (", ".join(pk_cols) + ", ") if pk_cols else ""

        #TAGS
        tag_lines = []
        seen_tags = set()
        for _, row in dfe.iterrows():
            raw_tag = (row.get(P['COLUMN_TAG']) or "").strip()
            tgt_col = (row.get(P['COLUMN_TGT_NAME']) or row.get(P['COLUMN_SRC_NAME']) or "").strip()
            
            if raw_tag and tgt_col:
                key = (tgt_col, raw_tag)
                if key in seen_tags:
                    continue
                seen_tags.add(key)
                tag_db, tag_rest = raw_tag.split(".", 1)
                dev_tag_db, prod_tag_db = db_to_env_dbs(tag_db)
                tag_db_expr = f"'{prod_tag_db}' if target.name == 'prod' else '{dev_tag_db}'"
                tag_lines.append(
                    f""""alter table {{{{ this }}}} modify column {tgt_col} set tag {{{{ {tag_db_expr} }}}}.{tag_rest}" """
                )

        tag_block = ""
        if tag_lines:
            tag_block = ",\n        " + ",\n        ".join(tag_lines)

        # --- Post-hook: per-company blocks (solo las empresas que tienen este entidad) ---
        company_table_pairs = []
        for schm_grp, grp in dfe.groupby(P['SCHM_SRC_NAME'], sort=True):
            asociado_nm = extract_string_part(str(schm_grp).strip().upper(), starts='BRONZE_', ends='_P').lower()
            tbl_nm      = str(grp[P['TABLE_SRC_NAME']].dropna().iloc[0])
            company_table_pairs.append((asociado_nm, tbl_nm))

        ph_lines = ["{%- set log_ref = source('support_param', 'INSERT_LOGS') -%}"]
        for a_nm, tbl_nm in company_table_pairs:
            ph_lines.append(f"{{% set src_ref_{a_nm} = source('bronze_{a_nm}', '{tbl_nm}') %}}")
            ph_lines.append(
                f"            update {{{{ log_ref }}}} l\n"
                f"            set tec_ts_integracion_b = current_timestamp()\n"
                f"            where l.tec_ts_integracion_b is null\n"
                f"              and l.start_watermark <> l.end_watermark\n"
                f"              and upper(l.database)   = upper('{{{{ src_ref_{a_nm}.database }}}}')\n"
                f"              and upper(l.schema)     = upper('{{{{ src_ref_{a_nm}.schema }}}}')\n"
                f"              and upper(l.table_name) = upper('{{{{ src_ref_{a_nm}.identifier }}}}')\n"
                f"              and exists (\n"
                f"                  select 1\n"
                f"                  from {{{{ ref('{entidad}_batch_{a_nm}') }}}} v\n"
                f"                  where v.tec_id_ingesta = l.tec_id_ingesta\n"
                f"              );"
            )
        post_hook_body = "\n".join(ph_lines)

        # Plantilla del modelo (con placeholders simples para no pelear con llaves de Jinja)
        tpl = textwrap.dedent("""\
{{ config(
    materialized='incremental',
    database=__DB__, schema='__SCHEMA__',
    incremental_strategy='append',
    tags=['silver','integ'],
    pre_hook=[
        "{% if is_incremental() %}
        -- (tec_cod_vigencia = 1 --> 0) Marcar como no vigentes registros con match y alguna diferencia. fec_negocio - 1 día (o mismo día si coincide inicio)
        merge into {{ this }} t
        using (
            select
                __PK_LIST__tec_des_empresa,
                __HASH_EXPR__ AS tec_hash,
                date(tec_ts_ingesta) as tec_fec_inicio
            from {{ ref('__ENTIDAD___batch') }}__SRC_WHERE__
        ) s
        on t.tec_cod_vigencia = 1
            and __PK_JOIN_UPDATE__
        when matched and t.tec_hash <> s.tec_hash then
            update set
                tec_cod_vigencia = 0,
                tec_fec_fin = case
                    when s.tec_fec_inicio = t.tec_fec_inicio then s.tec_fec_inicio
                    else dateadd(day, -1, s.tec_fec_inicio)
                end;
        {% endif %}"
    ],
    post_hook=[
        "__POST_HOOK_BODY____TAG_BLOCK__"
    ]
) }}

with source as (
    select
__BASE_SELECT__
    from {{ ref('__ENTIDAD___batch') }}__SRC_WHERE__
)
select
    s.*,
    to_date('2999-12-31')             as tec_fec_fin,
    1                                 as tec_cod_vigencia,
    current_timestamp()               as tec_ts_integracion_s
from source s
{% if is_incremental() %}
left join {{ this }} t
    on  __PK_JOIN_UPDATE__
    and t.tec_cod_vigencia    = 1
where __PK_NULL_CHECK__
    or t.tec_hash <> s.tec_hash
{% endif %}
        """)

        model = (tpl.replace("__DB__", db_expr)
                    .replace("__SCHEMA__", schm_tgt)
                    .replace("__ENTIDAD__", entidad_ref)
                    .replace("__SRC_TBL__", src_tbl)
                    .replace("__BASE_SELECT__", base_select)
                    .replace("__HASH_EXPR__", hash_expr)
                    .replace("__PK_LIST__", pk_list_prefix)
                    .replace("__PK_JOIN_UPDATE__", pk_join_update)
                    .replace("__PK_NULL_CHECK__", pk_null_check)
                    .replace("__SRC_SUFFIX__", src_suffix)
                    .replace("__POST_HOOK_BODY__", post_hook_body)
                    .replace("__TAG_BLOCK__", tag_block)
                    .replace("__SRC_WHERE__", src_where))

        out_path = os.path.join(out_dir, f"{tname}.sql")
        write_text(out_path, model)
        outputs.append(out_path)

    return outputs

    
write_integration_models(df)

In [ ]:
def write_stg_schema_yml(
    df: pd.DataFrame,
    path: str = "models/bronze/schema.yml",
    v_model_type: str = "INTEGRATION",
    starts_tbl_src: str = "V_ETL_", starts_schm_src: str = "BRONZE_", ends_schm_src: str = "_P"
):
    """
    Genera models/stg/schema.yml con tests para todas las vistas de batch:
    """
    # Nos quedamos con INTEGRATION
    dfi = df[df[P['MODEL_TYPE']].astype(str).str.upper().eq(v_model_type.upper())].copy()
    if dfi.empty:
        write_text(path, "version: 2\nmodels: []\n")
        return

    # Extraemos entidades (nombre canónico desde TABLE_TGT_NAME) y asociados
    dfi["__entidad__"] = dfi[P['TABLE_TGT_NAME']].astype(str).str.strip().apply(lambda x: extract_string_part(x))
    dfi["__asociado__"] = dfi[P['SCHM_SRC_NAME']].astype(str).str.strip().apply(lambda x: extract_string_part(x, starts=starts_schm_src, ends=ends_schm_src)).str.lower()

    # Nos quedamos con PKs y definimos su nombre con target
    pk_rows = dfi[dfi[P['COLUMN_PK']].apply(parse_bool)].copy()
    pk_rows["__pk_col__"] = pk_rows.apply(
        lambda r: (str(r.get(P['COLUMN_TGT_NAME']) or r.get(P['COLUMN_SRC_NAME']) or "").strip() or "").upper(), axis=1
    )

    # Diccionario que lista PKs únicas por entidad
    pk_map = (
        pk_rows.groupby("__entidad__")["__pk_col__"]
        .apply(lambda s: [c for c in dict.fromkeys([x for x in s.tolist() if x])])
        .to_dict()
    )

    # YAML
    lines = []
    lines.append("version: 2")
    lines.append("models:")

    # Por cada combinación entidad × asociado para la que exista parametría
    pairs_df = (dfi[["__entidad__", "__asociado__"]].dropna().drop_duplicates().sort_values(["__entidad__", "__asociado__"]))
    for _, row in pairs_df.iterrows():
        entidad  = row["__entidad__"]
        asociado = row["__asociado__"]
        model_name = f"{entidad}_batch_{asociado}".lower()
        entidad_desc = entidad.upper()
        asociado_desc = asociado.capitalize()

        lines.append(f"  - name: {model_name}")
        lines.append(f"    description: \"Batch pendiente más antiguo de {entidad_desc} ({asociado_desc}).\"")
        lines.append(f"    tests:")
        lines.append(f"      - no_nulls_except:")
        lines.append(f"          exclude: [\"tec_ts_integracion_b\"]")
        lines.append(f"      - single_batch:")
        lines.append(f"          column_name: tec_ts_ingesta")

        # En caso de que solo hubiese una PK en la entidad:
        pks = pk_map.get(entidad, [])
        if len(pks) == 1:
            # Un único PK -> unique
            lines.append(f"    columns:")
            lines.append(f"      - name: {yaml_safe_name(pks[0])}")
            lines.append(f"        tests: [unique]")

    # Guardia: si no se añadió ningún modelo, escribir lista vacía
    if len(lines) <= 2:
        write_text(path, "version: 2\nmodels: []\n")
        return

    content = "\n".join(lines) + "\n"
    write_text(path, content)

write_stg_schema_yml(df)



In [ ]:
def write_silver_schema_yml(
    df: pd.DataFrame,
    path: str = "models/silver/schema.yml",
    v_model_type: str = "INTEGRATION",
    starts_tbl_src: str = "V_ETL_",
):
    """
    Genera models/silver/schema.yml con:
      - tests para vistas agregadas <entidad>_batch (no_stale_batch_vs_integ)
      - tests para tablas SCD2 <entidad>_integ (scd2_* + no_nulls_except + accepted_values en AUD_CURRENT_IN)

    Requiere utilidades ya cargadas: extract_string_part, parse_bool, write_text.
    """
    dfi = df.copy()
    dfi = dfi[dfi[P['MODEL_TYPE']].astype(str).str.upper().eq(v_model_type.upper())].copy()
    if dfi.empty:
        write_text(path, "version: 2\nmodels: []\n")
        return

    # Entidad desde TABLE_TGT_NAME (nombre canónico Silver, igual para todas las empresas)
    dfi["__entidad__"] = dfi[P['TABLE_TGT_NAME']].astype(str).str.strip().apply(lambda x: extract_string_part(x))

    # Nos quedamos con PKs y definimos su nombre con target
    pk_rows = dfi[dfi[P['COLUMN_PK']].apply(parse_bool)].copy()
    pk_rows["__pk_col__"] = pk_rows.apply(
        lambda r: (str(r.get(P['COLUMN_TGT_NAME']) or r.get(P['COLUMN_SRC_NAME']) or "").strip() or "").upper(), axis=1
    )

    # Diccionario que lista PKs únicas por entidad
    pk_map = (
        pk_rows.groupby("__entidad__")["__pk_col__"]
        .apply(lambda s: [c for c in dict.fromkeys([x for x in s.tolist() if x])])
        .to_dict()
    )

    # YAML
    lines = []
    lines.append("version: 2")
    lines.append("models:")

    # Para cada entidad
    entidades = (dfi["__entidad__"].dropna().drop_duplicates().sort_values().tolist())
    for entidad in entidades:
        entidad_desc = entidad.upper()
        pks = pk_map.get(entidad, [])
        pks_yaml = json.dumps(pks)  # ["COL1", "COL2"] — comillas dobles, YAML válido

        # --- Bloque para <entidad>_batch ---
        # Solo tiene sentido si hay pk_cols para comparar con la integ
        if pks:
            lines.append(f"  - name: {entidad}_batch")
            lines.append(f"    description: \"Batch pendiente más antiguo de {entidad_desc} (todas las empresas).\"")
            lines.append( "    tests:")
            lines.append( "      - no_stale_batch_vs_integ:")
            lines.append(f"          integ_ref: {entidad}_integ")
            lines.append(f"          pk_cols: {pks_yaml}")

        # --- Bloque para <entidad>_integ ---
        if pks:
            model_integ = f"{entidad}_integ"
            lines.append(f"  - name: {model_integ}")
            lines.append(f"    description: \"Tabla SCD2 integrada de {entidad_desc}.\"")
            lines.append( "    tests:")
            lines.append( "      - scd2_single_current:")
            lines.append(f"          pk_cols: {pks_yaml}")
            lines.append( "      - scd2_no_overlap:")
            lines.append(f"          pk_cols: {pks_yaml}")
            lines.append( "      - no_nulls_except:")
            lines.append( "          exclude: []")
            lines.append( "    columns:")
            lines.append( "      - name: tec_cod_vigencia")
            lines.append( "        tests:")
            lines.append( "          - accepted_values:")
            lines.append( "              values: [0, 1]")

    # Guardia: si no se añadió ningún modelo, escribir lista vacía
    if len(lines) <= 2:
        write_text(path, "version: 2\nmodels: []\n")
        return

    content = "\n".join(lines) + "\n"
    write_text(path, content)

write_silver_schema_yml(df)

